# 📗 FastAPI 기초: 도서관 안내 봇 API 만들기

<div class="day77-callout" style="margin:10px 0;padding:8px 14px;font-size:17px;line-height:1.75;color:#202124;background:#ffffff;border-left:4px solid #54769b;border-radius:4px;"><strong style="display:block;font-size:1.13em;color:#1f3a5f;margin-bottom:2px;">실습 환경</strong><strong style="font-size:1em;color:#1f3a5f;">실행:</strong> 내 PC에서 FastAPI 서버와 노트북을 실행합니다. VS Code의 로컬 창을 사용합니다.<br><strong style="font-size:1em;color:#1f3a5f;">요청:</strong> 로컬 노트북이 같은 PC의 FastAPI 서버(127.0.0.1:8080)에 요청을 보내고, 브라우저에서 http://127.0.0.1:8080/docs를 직접 엽니다.<br><strong style="font-size:1em;color:#1f3a5f;">준비:</strong> 실습자료 폴더에서 uv sync로 환경을 구축하고, 실습자료/.venv를 노트북 커널로 선택합니다.<br><strong style="font-size:1em;color:#1f3a5f;">실습 자원:</strong> GPU와 vLLM은 쓰지 않습니다. DB 예시는 교안 폴더의 SQLite 파일 library.db를 쓰고(가입과 설정 없음), LangChain 스트리밍은 OPENAI_API_KEY를 실습자료/.env에서 읽습니다.</div>

#### **백엔드(backend)**

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">사용자가 보는 화면(프론트엔드) 뒤에서 요청을 받아 데이터를 조회하고, 계산하거나 모델을 부른 뒤 결과를 돌려주는 서버 쪽 프로그램</strong>입니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">필요한 이유:</strong> 비밀 키, 데이터베이스, GPU 모델은 사용자 기기에 둘 수 없습니다. 화면은 요청을 보내고 결과를 보여 주기만 하고, 키 확인과 데이터 조회와 무거운 계산은 서버 한곳에서 맡습니다. 그래서 웹 화면, 모바일 앱, 다른 프로그램이 같은 기능을 같은 규칙으로 함께 씁니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">쓰이는 곳:</strong> 쇼핑몰의 주문과 결제, 은행 앱의 잔액 조회, 챗봇의 답변 생성처럼 화면 뒤에서 데이터와 계산을 맡는 거의 모든 서비스에 있습니다. AI 서비스에서도 앱은 모델 서버를 직접 부르지 않고 백엔드의 API를 부르며, 백엔드가 인증과 사용량 기록을 한 뒤 모델 서버에 요청합니다. FastAPI는 이런 백엔드의 API를 파이썬으로 만드는 웹 프레임워크입니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="../images/05_backend_concept.png" width="1050" alt="왼쪽 사용자 기기의 웹 브라우저, 모바일 앱, 다른 프로그램이 요청을 보내면 가운데 서버의 백엔드(FastAPI)가 요청 확인, 데이터 조회, 계산과 모델 호출을 하고 JSON으로 응답합니다. 오른쪽의 데이터베이스와 모델 서버(GPU)는 백엔드만 부르고, 비밀 키도 서버에만 둡니다.">

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">이 교안에서는 도서관 안내 봇 API를 서버 파일 여섯 개(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">step1_endpoint.py</code>~<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">step6_langchain_stream.py</code>)로 한 단계씩 만들면서 FastAPI의 기본 기능을 익힙니다. 파일마다 로컬 터미널에서 웹 서버 프로그램 uvicorn으로 서버를 띄우고, 로컬 노트북에서 HTTP 요청 라이브러리 httpx로 요청을 보내 응답을 확인합니다. 교안 01에서는 같은 기능으로 vLLM 서버 앞에 상담 서빙 API를 세웁니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="../images/08_lesson00_goals.png" width="1050" alt="이번 교안의 학습 목표 네 가지를 번호 순서로 놓은 판 네 개입니다. 엔드포인트와 매개변수, 요청과 응답 스키마, def와 async def, 스트리밍 응답을 보여 줍니다.">

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">

<span class="day77-goal" style="font-size:1.13em;color:#1f3a5f;">**학습 목표**</span> (그림의 번호와 같습니다)

1. FastAPI 앱에 GET 엔드포인트를 만들고, 경로·쿼리 매개변수의 값을 검사하며, CORS로 허용할 웹 출처를 정합니다.
2. pydantic 모델로 POST 요청 본문과 응답 양식을 정하고, 양식이 틀린 요청이 422로 돌아오는 것을 확인합니다.
3. `def`와 `async def` 엔드포인트가 실행되는 곳(스레드 풀, 이벤트 루프)을 알고, 요청 4건을 동시에 보내 `async def` 안의 `time.sleep`이 다른 요청까지 멈추는 것을 확인합니다.
4. StreamingResponse로 모델 응답을 중계하고, lifespan으로 서버 시작·종료 때 DB·HTTP 연결을 준비·정리합니다.

</div>

## 1. 실습 준비

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">이 교안에서 계속 쓰는 웹 API 용어입니다.</p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">API 서버(API server):</strong> 다른 프로그램이 정해진 주소와 양식으로 요청을 보내면 처리 결과를 돌려주는 프로그램입니다. 이 교안의 FastAPI 서버가 API 서버입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">HTTP 요청(HTTP request)과 응답(HTTP response):</strong> 요청은 메서드(요청의 종류), 주소, 필요하면 요청 본문으로 이루어집니다. GET은 조회할 때, POST는 본문에 담은 데이터의 처리를 맡길 때 씁니다. 응답은 상태 코드와 응답 본문으로 이루어집니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">JSON(JavaScript Object Notation):</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">{&quot;키&quot;: 값}</code> 모양으로 데이터를 적는 글자 형식입니다. FastAPI는 함수가 돌려준 딕셔너리를 JSON으로 바꿔 보내고, httpx의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">response.json()</code>은 받은 JSON을 딕셔너리로 되돌립니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">상태 코드(status code):</strong> 서버가 요청을 어떻게 처리했는지 알리는 세 자리 숫자입니다. 200번대는 성공, 400번대는 요청 쪽 문제, 500번대는 서버 쪽 문제입니다.</li>
</ul>
</div>

### 1-1. 터미널과 노트북의 역할

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">실습자료</code> 폴더(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">pyproject.toml</code>이 있는 폴더)에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">uv sync</code>로 환경을 구축한 뒤 진행합니다. VS Code 로컬 창에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">실습자료</code>를 열고, 노트북 커널은 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">실습자료/.venv</code></strong>를 선택합니다.</p>
<p style="margin:6px 0;">서버를 실행할 때는 탐색기에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">교안_00_FastAPI_기초</code> 폴더를 오른쪽 클릭해 <strong style="font-size:1.13em;color:#1f3a5f;">Open in Integrated Terminal</strong>을 선택합니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">uv run</code> 명령이 공용 환경을 사용하므로 별도 활성화 명령은 필요하지 않습니다.</p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">작업</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">실행할 곳</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">역할</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">uvicorn</code>으로 단계별 FastAPI 서버 실행과 종료(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code>)</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">로컬 터미널</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">교안 폴더에서 서버 파일 하나를 실행합니다. 노트북은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">127.0.0.1:8080</code>으로 접속합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">요청 보내기, 응답과 걸린 시간 확인</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">로컬 노트북</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">FastAPI 서버에 httpx로 요청하는 클라이언트입니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">서버 파일(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">step*.py</code>) 읽기, 함께 따라하기 빈칸 채우기</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">VS Code 편집기</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">파일을 저장하면 로컬 터미널의 서버가 다시 시작합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">API 문서 화면(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/docs</code>) 열기</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">내 PC 브라우저</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">http://127.0.0.1:8080/docs</code>를 직접 엽니다.</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;">학생용 서버 파일은 교안 폴더에, 완성본은 그 안의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">정답/</code>에 있습니다. 함께 따라하기의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">...</code>는 VS Code 편집기에서 채우고 저장합니다. 서버는 한 번에 step 파일 하나만 실행합니다.</p>
</div>

### 1-2. 노트북 준비

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">httpx</strong>는 파이썬에서 HTTP 요청을 보내는 라이브러리입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">httpx.get</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">httpx.post</code>로 요청 하나를 보내고, 비동기 요청과 스트리밍 응답 읽기도 지원합니다. 이 교안은 vLLM에 요청하지 않으므로 준비 셀은 FastAPI 서버 주소와 응답을 보는 함수 하나만 둡니다.</p>
</div>

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 로컬 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">내 PC의 실습자료/.venv 커널에서 실행합니다</span></div>

In [ ]:
import httpx
import pandas as pd

# 로컬 터미널에서 띄운 FastAPI 서버 주소(내 PC의 8080 포트)
SERVING_URL = "http://127.0.0.1:8080"

# 테이블의 긴 글(안내문, 답변)이 잘리지 않게 칸의 글자 수 제한을 풉니다
pd.set_option("display.max_colwidth", None)


def status_and_body(response):
    """응답의 상태 코드와 본문(JSON을 딕셔너리로 되돌린 값)을 딕셔너리 하나로 모읍니다."""
    return {"상태 코드": response.status_code, "본문": response.json()}

## 2. 첫 엔드포인트와 매개변수

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">FastAPI</strong>는 파이썬 함수에 주소를 붙여 웹 API로 만드는 라이브러리입니다. 함수 위에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">@app.get(&quot;/주소&quot;)</code> 한 줄을 붙이면 그 함수가 엔드포인트가 됩니다. FastAPI는 매개변수의 <strong style="font-size:1.13em;color:#1f3a5f;">타입 힌트(type hint)</strong>를 읽어, 받은 값을 그 자료형으로 바꾸고 검사합니다. 타입 힌트는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">book_id: int</code>처럼 매개변수 이름 뒤에 적은 자료형입니다. <strong style="font-size:1.13em;color:#1f3a5f;">uvicorn</strong>은 FastAPI 앱을 불러와 실제로 요청을 받는 웹 서버 프로그램입니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">엔드포인트(endpoint)</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">요청을 받는 주소와 HTTP 메서드(요청 방식. GET, POST)의 짝, 그리고 그 요청을 처리하는 함수입니다. 예를 들어 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">GET /books/12</code> 요청은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">get_book</code> 함수가 처리하고 책 정보를 JSON으로 돌려줍니다.</li>
<li style="margin:3px 0;">백엔드의 API는 엔드포인트의 모음입니다. 주소, 메서드, 받는 값의 양식을 정해 두면 다른 프로그램은 서버 안의 코드를 몰라도 그 주소로 요청해 결과를 받을 수 있습니다.</li>
<li style="margin:3px 0;">서버 상태 확인, 질문 보내기, 답을 조각으로 받기처럼 보통 서비스 기능 하나마다 엔드포인트를 하나씩 둡니다.</li>
</ul>
<img style="display:block;max-width:100%;height:auto;margin:0;" src="../images/07_endpoint_concept.png" width="1050" alt="GET /books/12 요청은 get_book 함수가, GET /notice?topic=대출 요청은 get_notice 함수가 받아 JSON으로 답하는 구조입니다. 주소 안의 값은 경로 매개변수, 주소 뒤의 조건은 쿼리 매개변수입니다.">
</div>

### 2-1. step1_endpoint.py 서버 실행

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">교안 폴더에서 연 로컬 터미널에서 아래 블록을 실행합니다. 서버가 실행되는 동안 이 터미널을 그대로 둡니다.</p>
</div>

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · 로컬 터미널</strong></div>

```text
pwd  # 끝이 교안_00_FastAPI_기초인지 확인합니다

# FastAPI 서버를 띄웁니다(멈출 때까지 이 터미널을 씁니다)
uv run uvicorn step1_endpoint:app --host 0.0.0.0 --port 8080 --reload
```

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">명령어 설정</strong></p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">설정</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">의미</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">uv run uvicorn step1_endpoint:app</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">지금 폴더의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">step1_endpoint.py</code>에 든 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">app</code> 객체를 서버로 띄웁니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">:</code> 앞은 모듈 이름(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.py</code>를 뺀 파일 이름), 뒤는 앱 변수 이름입니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--host 0.0.0.0</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">모든 IPv4 네트워크 인터페이스에서 요청을 받습니다. 같은 PC에서는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">127.0.0.1:8080</code>으로 접속합니다. 외부에서는 접근 가능한 IP와 공개 포트가 필요합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--port 8080</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">8080 포트에서 요청을 받습니다. 교안 01에서 vLLM 서버가 8000을 쓰므로 다른 포트를 씁니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--reload</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">서버 코드(.py)를 저장하면 서버를 다시 시작합니다. 함께 따라하기에서 빈칸을 채운 뒤 명령을 다시 실행하지 않아도 됩니다.</td>
</tr>
</tbody>
</table>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">실행 폴더:</strong> uvicorn은 명령을 실행한 폴더에서 모듈을 찾습니다. 학생 서버는 교안 폴더에서 빈칸판 step 파일을 띄웁니다. 정답 노트북의 명령은 교안 폴더의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">정답/</code>으로 이동해 완성본 step 파일을 띄웁니다.</li>
</ul>
</div>

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">로컬 터미널에 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Application startup complete</code></strong>와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Uvicorn running on http://0.0.0.0:8080</code>이 나오면 요청을 보낼 수 있습니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--reload</code>를 붙였으므로 서버 파일을 저장하면 서버가 다시 시작합니다.</p>
</div>

### 2-2. 앱과 상태 확인 엔드포인트

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">FastAPI(...)</code>가 앱 객체 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">app</code>을 만듭니다. <strong style="font-size:1.13em;color:#1f3a5f;">데코레이터(decorator)</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">@app.get(&quot;/healthz&quot;)</code>는 바로 아래 함수를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">GET /healthz</code> 요청을 처리하는 함수로 등록합니다. <strong style="font-size:1.13em;color:#1f3a5f;">상태 확인(health check)</strong> 엔드포인트는 서버가 요청을 받을 수 있는지만 알려 주며, 배포한 서버를 자동으로 점검하는 프로그램이 이 주소에 요청합니다.</p>
</div>

```python
# step1_endpoint.py
# title, description, version 은 /docs 문서 화면 맨 위에 보인다
app = FastAPI(
    title="도서관 안내 봇 API",
    description="도서관 이용 문의에 답하는 서비스입니다.",
    version="0.1.0",
)


# GET /healthz 요청을 이 함수가 처리한다. 주소를 정하는 것은 함수 이름이 아니라 데코레이터의 "/healthz" 다
@app.get("/healthz")
def health_check():
    """서버가 요청을 받을 수 있으면 상태 ok 를 돌려줍니다."""
    # 딕셔너리를 돌려주면 FastAPI 가 JSON 으로 바꿔 보낸다
    return {"status": "ok", "service": "library-bot"}
```

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 로컬 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">내 PC의 실습자료/.venv 커널에서 실행합니다</span></div>

In [ ]:
# 상태 확인 주소에 GET 요청을 보냅니다
health_response = httpx.get(f"{SERVING_URL}/healthz")

# 상태 코드와 응답 본문을 함께 봅니다
health_check = status_and_body(health_response)
display(health_check)

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">상태 코드 200, 본문 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">{'status': 'ok', 'service': 'library-bot'}</code>:</strong> 로컬 터미널의 FastAPI 서버가 요청을 받았고, 함수가 돌려준 딕셔너리가 JSON으로 왔습니다. 로컬 터미널에도 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;GET /healthz HTTP/1.1&quot; 200 OK</code> 줄이 찍힙니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">ConnectError</code>가 나면:</strong> 로컬 터미널의 서버가 떠 있지 않은 것입니다. 위 실행 명령과 로컬 터미널의 오류 줄을 확인합니다.</li>
</ul>
</div>

### 2-3. 경로 매개변수와 404

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">경로 매개변수(path parameter)</strong>는 주소 안에 넣는 값입니다. 주소에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">{book_id}</code>처럼 중괄호로 자리를 만들고 함수 매개변수 이름을 같게 맞추면, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/books/12</code>의 12가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">book_id</code>로 들어옵니다. 책 한 권처럼 특정 대상을 가리키는 값을 경로 매개변수로 받습니다.</p>
</div>

```python
# step1_endpoint.py
# 책 번호 -> 제목, 서가 위치, 대출 가능 여부
BOOKS = {
    12: {"title": "파이썬으로 배우는 데이터 분석", "shelf": "A-3", "available": True},
    45: {"title": "도서관 이용 안내서", "shelf": "B-1", "available": False},
}


# 주소의 {book_id} 자리 값이 같은 이름의 매개변수로 들어온다(경로 매개변수)
@app.get("/books/{book_id}")
def get_book(book_id: int = Path(ge=1)):  # 글자 "12" 를 정수 12 로 바꿔 넘긴다. Path(ge=1) 은 1 이상만 받는다
    """책 번호로 책 한 권의 정보를 돌려줍니다. 없는 번호면 404 입니다."""
    book = BOOKS.get(book_id)

    # 없는 책이면 함수를 여기서 끝내고 404(Not Found, 요청한 대상 없음)와 {"detail": 메시지} 본문을 돌려준다
    if book is None:
        raise HTTPException(status_code=404, detail=f"{book_id}번 책이 없습니다.")

    # 책 번호와 책 정보(title, shelf, available)를 한 딕셔너리로 합친다
    return {"book_id": book_id, **book}
```

In [ ]:
# 12번 책의 정보를 요청합니다
book_response = httpx.get(f"{SERVING_URL}/books/12")
book_12 = status_and_body(book_response)
display(book_12)

In [ ]:
# BOOKS에 없는 99번 책을 요청합니다
missing_response = httpx.get(f"{SERVING_URL}/books/99")
missing_book = status_and_body(missing_response)
display(missing_book)

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">12번 책은 200과 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">'shelf': 'A-3'</code>:</strong> 함수가 돌려준 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">book_id</code>와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">BOOKS[12]</code>의 세 항목이 한 딕셔너리로 왔습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">99번 책은 404와 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">{'detail': '99번 책이 없습니다.'}</code>:</strong> 200에 &quot;없음&quot;을 적어 보낼 수도 있지만, 상태 코드로 알리면 요청한 프로그램이 본문을 읽지 않고도 실패를 압니다. httpx의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">response.raise_for_status()</code>도 400번대, 500번대 상태 코드에서 오류를 냅니다.</li>
</ul>
</div>

### 2-4. 쿼리 매개변수

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">쿼리 매개변수(query parameter)</strong>는 주소 뒤에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">?이름=값&amp;이름=값</code>으로 붙이는 값입니다.</p>
<p style="margin:6px 0;">실무에서는 <strong style="font-size:1.13em;color:#1f3a5f;">어떤 대상에 대한 요청인지는 경로에, 그 대상을 어떤 조건으로 조회하거나 보여 줄지는 쿼리에 둡니다.</strong></p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">사용할 때</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">선택</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">실무 예</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">특정 대상 하나를 식별</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">경로 매개변수</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">책 번호 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/books/12</code>, 사용자 ID <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/users/7</code>, 주문 번호 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/orders/1004</code></td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">목록을 검색·필터링하거나 정렬·페이지를 지정</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">쿼리 매개변수</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">검색어 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">?keyword=파이썬</code>, 필터 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">?category=IT</code>, 정렬 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">?sort=price</code>, 페이지 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">?page=2</code></td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">같은 대상의 응답 언어·표시 길이 등을 지정</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">쿼리 매개변수</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">언어 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">?lang=ko</code>, 최대 길이 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">?max_length=15</code></td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;">둘을 함께 쓰면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/users/7/orders?status=paid&amp;page=2</code>처럼 됩니다. <strong style="font-size:1.13em;color:#1f3a5f;">7번 사용자의 주문 목록</strong>은 경로로, <strong style="font-size:1.13em;color:#1f3a5f;">결제 완료 주문만 2페이지로 조회</strong>하는 조건은 쿼리로 정했습니다.</p>
<p style="margin:6px 0;">이 교안의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/books/12</code>에서는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">book_id</code>가 책 한 권을 가리키므로 경로로 받습니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/notice?topic=대출&amp;max_length=15</code>의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">topic</code>은 안내문을 고르는 조건이고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">max_length</code>는 표시 길이이므로 쿼리로 받습니다.</p>
</div>

```python
# step1_endpoint.py
# 주제 -> 안내문
NOTICES = {
    "대출": "도서 대출은 1인 5권까지, 대출 기간은 14일입니다.",
    "열람실": "열람실은 평일 9시부터 22시까지, 주말은 9시부터 18시까지 운영합니다.",
}


# 주소의 {} 에 없는 매개변수는 쿼리 매개변수(?topic=...&max_length=...)가 된다. 기본값이 있으면 보내지 않아도 된다
# Query(default=100, ge=1, le=500): 보내지 않으면 100, 1 이상 500 이하만 받는다
@app.get("/notice")
def get_notice(topic: str, max_length: int = Query(default=100, ge=1, le=500)):
    """주제별 안내문을 돌려줍니다. max_length 로 안내문 길이를 자를 수 있습니다."""
    text = NOTICES.get(topic, "해당 주제의 안내문이 없습니다.")
    return {"topic": topic, "notice": text[:max_length], "truncated": len(text) > max_length}
```

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Path</code>는 경로 값, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Query</code>는 쿼리 값의 제약을 선언합니다.</p>
</div>

In [ ]:
# topic만 보냅니다(max_length는 기본값 100이 쓰입니다). params 딕셔너리는 주소 뒤에 ?topic=... 형식으로 붙습니다
loan_response = httpx.get(f"{SERVING_URL}/notice", params={"topic": "대출"})
loan_notice = status_and_body(loan_response)
display(loan_notice)

In [ ]:
# topic과 max_length를 함께 보내고, httpx가 실제로 보낸 주소도 함께 봅니다
room_response = httpx.get(f"{SERVING_URL}/notice", params={"topic": "열람실", "max_length": 15})
room_notice = status_and_body(room_response)
room_notice["보낸 주소"] = str(room_response.url)
display(room_notice)

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">truncated</code>:</strong> 대출 안내문은 100자보다 짧아 False이고, 열람실 안내문은 15자에서 잘려 True입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">보낸 주소</code>:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">?topic=...&amp;max_length=15</code> 모양으로 조건이 주소 뒤에 붙었습니다. httpx는 한글 값을 주소에 쓸 수 있는 글자(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">%EC%97%B4...</code>)로 바꿔 보냅니다.</li>
</ul>
</div>

### 2-5. 자료형과 입력 제약, 422

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">FastAPI는 타입 힌트대로 값을 바꾸고, 바꿀 수 없거나 필수 값이 빠지거나 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Path</code>·<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Query</code>의 범위를 벗어나면 함수를 부르지 않고 <strong style="font-size:1.13em;color:#1f3a5f;">422(Unprocessable Content)</strong>를 돌려줍니다. 422 응답 본문은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">{&quot;detail&quot;: [{&quot;type&quot;: ..., &quot;loc&quot;: [...], &quot;msg&quot;: ..., &quot;input&quot;: ...}]}</code> 모양입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">detail</code>은 틀린 값마다 딕셔너리 하나를 담은 리스트이고, 딕셔너리에는 오류 종류(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">type</code>), 위치(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">loc</code>), 이유(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">msg</code>), 받은 값(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">input</code>)이 들어 있습니다. 아래 셀은 리스트의 첫 딕셔너리(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">[&quot;detail&quot;][0]</code>)에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">loc</code>과 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">msg</code>를 꺼냅니다. 로컬 터미널의 요청 기록에는 같은 코드의 옛 이름인 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">422 Unprocessable Entity</code>로 찍힙니다.</p>
</div>

In [ ]:
# 타입 오류, 필수 값 누락, 범위 위반 요청을 보냅니다
bad_requests = {
    "GET /books/abc": httpx.get(f"{SERVING_URL}/books/abc"),
    "GET /notice?topic=대출&max_length=길게": httpx.get(f"{SERVING_URL}/notice", params={"topic": "대출", "max_length": "길게"}),
    "GET /notice (topic 없음)": httpx.get(f"{SERVING_URL}/notice"),
    "GET /books/0 (1 미만)": httpx.get(f"{SERVING_URL}/books/0"),
    "GET /notice?topic=대출&max_length=501": httpx.get(f"{SERVING_URL}/notice", params={"topic": "대출", "max_length": 501}),
}

# 422 본문의 detail 목록에서 첫 항목의 위치(loc)와 이유(msg)를 꺼내 한 테이블로 봅니다
error_rows = []
for label, response in bad_requests.items():
    first_error = response.json()["detail"][0]
    error_rows.append({"요청": label, "상태 코드": response.status_code, "loc": first_error["loc"], "msg": first_error["msg"]})
display(pd.DataFrame(error_rows))

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">loc</code>:</strong> 첫 칸은 값이 온 곳(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">path</code>는 경로 매개변수, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">query</code>는 쿼리 매개변수), 둘째 칸은 매개변수 이름입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">msg</code>:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">unable to parse string as an integer</code>는 정수로 바꿀 수 없다는 뜻이고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Field required</code>는 필수 값이 빠졌다는 뜻입니다. 함수 안에 검사 코드가 한 줄도 없어도 타입 힌트, 기본값, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Path</code>·<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Query</code>의 제약으로 이 검사가 이루어집니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">범위 위반:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/books/0</code>은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ge=1</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">max_length=501</code>은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">le=500</code>에 걸려 422가 됩니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ge</code>는 이상, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">le</code>는 이하입니다.</li>
</ul>
<p style="margin:6px 0;">이 교안에서 만나는 상태 코드입니다.</p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">상태 코드</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">뜻</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">이 교안의 예</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">200 OK</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">정상 처리</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">GET /books/12</code></td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">404 Not Found</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">요청한 대상이 없음(함수가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">HTTPException</code>으로 알림)</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">GET /books/99</code></td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">422 Unprocessable Content</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">요청 값이 타입 힌트나 양식 규칙에 맞지 않음(함수를 부르지 않음)</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">GET /books/abc</code></td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">500 Internal Server Error</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">서버 코드 안에서 처리하지 못한 오류</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">함께 따라하기의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">BOOKS[77]</code>이 아직 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">...</code>일 때 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">GET /books/77</code></td>
</tr>
</tbody>
</table>
</div>

### 2-6. 🖐️ 함께 따라하기: 책 더하기와 언어 선택 매개변수

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">본실습에서 본 경로 매개변수와 쿼리 매개변수를 직접 써서 같은 서버 파일에 책 한 권과 주소 하나를 더하고, 노트북에서 요청해 확인합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">본실습과 다른 점</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">서버 파일:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">step1_endpoint.py</code> 맨 아래에서 책 정보를 추가하고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">get_notice_v2</code>의 인자·데코레이터·본문을 작성합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">책:</strong> 77번 책(제목 &quot;FastAPI로 만드는 서빙 API&quot;, 서가 &quot;C-2&quot;, 대출 가능)을 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">BOOKS</code>에 더합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">주소:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">GET /v2/notice</code>. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/notice</code>를 고치면 그 주소를 이미 쓰는 프로그램이 받는 응답 양식이 바뀌므로, 언어 선택 쿼리 매개변수 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">lang</code>(문자열, 기본값 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;ko&quot;</code>)을 더한 주소를 따로 엽니다. 응답에는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">topic</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">notice</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">truncated</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">lang</code> 네 항목을 담습니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">notice</code>는 주제에 맞는 안내문을 최대 길이로 자른 값, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">truncated</code>는 잘렸는지 여부이고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">lang</code>은 받은 값을 그대로 담습니다.</li>
</ul>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">작성 순서:</strong> (1) 책 정보를 추가합니다. (2) <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">get_notice_v2</code>에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">topic</code>(str, 필수), <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">max_length</code>(int, 기본값 100, Query로 1~500), <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">lang</code>(str, 기본값 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;ko&quot;</code>)을 선언하고 GET 주소를 등록한 뒤 본문을 작성합니다. 이름만 남긴 함수의 선언과 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">pass</code>를 고치고, 아래 요청 셀의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">...</code>도 채웁니다. 완성 코드는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">정답/step1_endpoint.py</code>와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">정답/교안_00_정답.ipynb</code>의 같은 절에 있습니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">설정</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">로컬 터미널의 step1 서버는 그대로 둡니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--reload</code>로 띄웠으므로 파일을 저장하면 서버가 다시 시작합니다.</li>
</ul>
<p style="margin:6px 0;">VS Code 편집기에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">step1_endpoint.py</code>를 열어 맨 아래 책 정보를 채우고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">get_notice_v2</code>를 완성한 뒤 저장합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">확인 순서:</strong> 파일을 저장하고 재시작 로그를 확인한 뒤 아래 요청 셀을 실행합니다. 404이면 데코레이터의 메서드·주소를 먼저 확인하고, 422이면 요청 필드·타입·기본값을 확인합니다. 함수 내부 오류는 터미널의 Traceback을 확인합니다.</p>
</div>

```python
# step1_endpoint.py 맨 아래
# ---------------------------------------------------------------------------
# 🖐️ 함께 따라하기
# ---------------------------------------------------------------------------

# (1) 77번 책의 정보를 BOOKS에 추가하세요.
BOOKS[77] = ...


# (2) GET /v2/notice의 인자, 데코레이터, 본문을 작성하세요.
def get_notice_v2():
    pass
```

<details style="margin:8px 0;"><summary style="font-size:17px;color:#1f3a5f;font-weight:600;cursor:pointer;margin:6px 0;">힌트</summary>

```text
접근방법: 2-3의 BOOKS, 2-4의 get_notice를 참고합니다.
(1) BOOKS[77]에는 BOOKS의 다른 책과 같은 세 키(title, shelf, available)의 딕셔너리를 넣습니다.
(2) get_notice의 안내문 조회와 길이 제한을 참고하고, 응답 딕셔너리에 받은 lang 값도 담습니다.
```

</details>

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">확인 기준:</strong> 파일을 저장하면 로컬 터미널에 서버가 다시 시작하는 줄과 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Application startup complete</code>가 다시 나오고, 오류 줄(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Traceback</code>)이 없습니다.</p>
</div>

In [ ]:
# 🖐️ 함께 따라하기: (1)에서 BOOKS에 더한 77번 책을 요청합니다
new_book_id = 77
new_book_response = httpx.get(f"{SERVING_URL}/books/{new_book_id}")
# 200이 아니면 여기서 오류로 멈춥니다
new_book_response.raise_for_status()
new_book = status_and_body(new_book_response)
display(new_book)

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">확인 기준:</strong> 상태 코드 200이고 본문의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shelf</code>가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">C-2</code>입니다. 서버 파일의 빈칸이 남아 있으면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">HTTPStatusError</code>(500 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Internal Server Error</code>)가 나고 로컬 터미널에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Traceback</code>이 찍힙니다.</p>
</div>

In [ ]:
# 🖐️ 함께 따라하기: 두 번째 요청의 params에 topic "대출"과 lang "en"을 담으세요
lang_params = [{"topic": "대출"}, ...]

# 요청마다 상태 코드와 응답의 lang 값을 한 테이블로 봅니다
lang_rows = []
for params in lang_params:
    response = httpx.get(f"{SERVING_URL}/v2/notice", params=params)
    # 200이 아니면 여기서 오류로 멈춥니다
    response.raise_for_status()
    lang_rows.append({"보낸 params": params, "상태 코드": response.status_code, "lang": response.json()["lang"]})
display(pd.DataFrame(lang_rows))

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">확인 기준:</strong> 두 행 모두 상태 코드 200이고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">lang</code>을 보내지 않은 첫 행은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ko</code>, 둘째 행은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">en</code>입니다.</p>
</div>

### 2-7. CORS: 웹 화면의 출처 허용

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">CORS(Cross-Origin Resource Sharing, 교차 출처 리소스 공유)</strong>는 브라우저에 띄운 웹 페이지가 <strong style="font-size:1.13em;color:#1f3a5f;">출처가 다른 API</strong>의 응답을 읽어도 되는지를 API 서버가 정해 주는 설정입니다. 출처는 웹 주소의 앞부분(프로토콜, 호스트, 포트)이라서, 웹 페이지 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">http://localhost:3000</code>과 이 API <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">http://127.0.0.1:8080</code>은 출처가 다릅니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">왜 있는 규칙인가</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">브라우저는 기본으로 &quot;웹 페이지는 자기와 출처가 같은 API의 응답만 읽는다&quot;는 규칙을 지킵니다. 사이트를 열면 그 사이트의 페이지가 사용자의 브라우저 안에서 실행되기 때문입니다.</li>
<li style="margin:3px 0;">예: 사용자가 은행 사이트에 로그인해 둔 채 광고 사이트를 엽니다. 광고 페이지가 사용자의 브라우저로 은행 API에 잔액 조회를 보내면, 브라우저는 로그인 정보를 함께 실어 보냅니다. 이 응답을 광고 페이지가 읽을 수 있다면 사용자의 잔액이 광고 사이트로 넘어갑니다. 브라우저의 기본 규칙이 바로 이 응답 읽기를 막습니다.</li>
<li style="margin:3px 0;">CORS는 이 규칙에 예외를 만듭니다. API 서버가 허락한 출처(우리 서비스의 웹 페이지)만 응답을 읽게 하고, 허락한 출처는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Access-Control-Allow-Origin</code> 응답 헤더로 브라우저에 알립니다.</li>
</ul>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">CORS 설정</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">결과</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">설정 없음</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">안전합니다. 다만 출처가 다른 우리 웹 페이지도 이 API의 응답을 읽지 못합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">우리 웹 페이지의 출처만 허용</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">올바른 설정입니다. 우리 페이지는 쓰고, 다른 사이트의 페이지는 읽지 못합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">모든 출처 허용</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">위험합니다. 아무 사이트의 페이지나 방문자의 브라우저를 거쳐 응답을 읽어 가므로, 로그인한 사용자의 정보가 그 사이트로 새어 나갈 수 있습니다.</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;">CORS는 브라우저만 지키는 규칙이라 노트북의 httpx나 curl 같은 프로그램의 요청에는 적용되지 않습니다. 누가 API를 쓸 수 있는지는 CORS가 아니라 API 키 같은 인증으로 막습니다.</p>
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">CORSMiddleware</code>는 이 CORS 헤더를 응답에 붙이고 사전 요청을 처리하는 미들웨어입니다. step1은 다음 설정으로 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">http://localhost:3000</code> 출처와 GET 메서드를 허용합니다.</p>
</div>

```python
from fastapi.middleware.cors import CORSMiddleware
```

```python
# step1_endpoint.py
# 출처(origin)는 주소의 프로토콜(http/https), 호스트(localhost 등), 포트를 합친 것이다.
# 예를 들어 http://localhost:3000에서 연 웹 화면의 JavaScript가 이 API를 호출한다고 가정한다.
# API 주소는 http://127.0.0.1:8080이므로 호스트와 포트가 달라 서로 다른 출처다.
# allow_origins에는 API 주소가 아니라, API를 호출하는 웹 화면의 출처를 적는다.
# 이 설정은 브라우저가 허용된 웹 화면에서 API 응답을 읽을 수 있게 한다.
app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:3000"],
    allow_methods=["GET"],
)
```

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">브라우저는 일부 교차 출처 요청을 보내기 전에 <strong style="font-size:1.13em;color:#1f3a5f;">사전 요청(preflight)</strong>을 OPTIONS 메서드로 보내 허용 여부를 먼저 확인합니다. 아래 코드는 httpx로 이 사전 요청을 흉내 내어 서버의 응답을 비교합니다.</p>
</div>

In [ ]:
# 허용한 웹 출처와 허용하지 않은 출처의 사전 요청을 비교합니다.
cors_rows = []
for origin in ["http://localhost:3000", "http://localhost:3001"]:
    response = httpx.options(
        f"{SERVING_URL}/notice",
        headers={"Origin": origin, "Access-Control-Request-Method": "GET"},
    )
    cors_rows.append({"웹 출처": origin, "상태 코드": response.status_code,
                      "허용 출처 헤더": response.headers.get("access-control-allow-origin")})
display(pd.DataFrame(cors_rows))

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기:</strong> 허용한 3000 포트는 200과 그 출처를 담은 헤더가, 3001 포트는 400과 빈 허용 출처 헤더(None)가 나옵니다.</p>
</div>

## 3. 요청 본문과 스키마

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">요청 본문(request body)</strong>은 주소가 아니라 요청 안에 담아 보내는 데이터입니다. 이 교안은 JSON 본문을 쓰며, 본문을 담는 요청은 보통 POST 메서드로 보냅니다. 주소는 짧아야 하고 브라우저 기록과 서버 로그에 그대로 남으므로, 긴 글(질문 문장), 항목이 여러 개인 값, 다른 사람에게 보이면 안 되는 값은 본문에 담습니다.</p>
</div>

### 3-1. step2_schema.py 서버 실행

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · 로컬 터미널</strong></div>

```text
# 앞 단계 서버를 Ctrl+C로 멈춘 뒤, 같은 로컬 터미널에서 실행합니다
uv run uvicorn step2_schema:app --host 0.0.0.0 --port 8080 --reload
```

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">로컬 터미널에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Application startup complete</code>가 다시 나오면 새 서버가 요청을 받을 수 있습니다. 로컬 터미널을 새로 열었다면 탐색기에서 교안 폴더의 터미널을 다시 엽니다. 정답 서버를 실행할 때는 이어서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">cd 정답</code>으로 이동합니다.</p>
</div>

### 3-2. pydantic BaseModel로 요청 양식 정하기

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">pydantic</strong>은 클래스에 항목 이름과 타입을 적어 데이터 양식을 정하고, 들어온 값을 그 양식으로 검사하는 라이브러리입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">BaseModel</code>을 물려받은 클래스(pydantic 모델)가 양식 하나이며, FastAPI에서는 이 양식을 <strong style="font-size:1.13em;color:#1f3a5f;">스키마(schema)</strong>라고도 부릅니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Field</code>는 항목에 기본값, 제약(값이 지켜야 할 조건. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">min_length</code> 최소 글자 수, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ge</code> 이상, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">le</code> 이하), 설명을 붙입니다.</p>
</div>

```python
# step2_schema.py
class AskRequest(BaseModel):
    """도서관 봇에게 보내는 질문 한 건."""

    # 기본값이 없는 항목은 필수, 기본값이 있는 항목은 보내지 않으면 기본값이 채워진다
    question: str = Field(min_length=2, description="이용자의 질문 문장")
    member_level: str = Field(default="basic", description="회원 등급(basic 또는 premium)")
    # ge 는 이상, le 는 이하. 10~500 밖의 값은 함수에 닿기 전에 422 로 돌아간다. 제약은 /docs 에도 그대로 표시된다
    max_length: int = Field(default=100, ge=10, le=500, description="답변 최대 길이")
```

### 3-3. 요청 양식을 엔드포인트에 붙이기

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">함수 매개변수의 타입을 pydantic 모델로 적으면, FastAPI가 요청 본문(JSON)을 그 양식으로 검사해 모델 객체로 넘깁니다. POST 요청을 받는 함수는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">@app.get</code> 대신 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">@app.post(&quot;/주소&quot;)</code>로 등록합니다. 함수 안에서는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">payload.question</code>처럼 점으로 항목을 꺼냅니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">payload</code>는 딕셔너리가 아니라 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">AskRequest</code> 객체라서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">payload[&quot;question&quot;]</code>처럼 대괄호로 꺼내면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">TypeError</code>가 나고 응답은 500이 됩니다. 함수가 불렸다면 검사는 이미 끝났으므로 타입 확인 코드를 쓰지 않습니다.</p>
</div>

```python
# step2_schema.py
# 질문에 든 낱말 -> 안내문
ANSWERS = {
    "대출": "도서 대출은 1인 5권까지, 대출 기간은 14일입니다.",
    "연장": "대출 연장은 예약자가 없을 때 1회 7일 가능합니다.",
    "열람실": "열람실은 평일 9시부터 22시까지 운영합니다.",
}


def find_answer(question: str) -> str:
    """질문에 들어 있는 낱말로 안내문을 고릅니다."""
    for keyword, text in ANSWERS.items():
        if keyword in question:
            return text
    return "문의하신 내용은 안내 데스크에서 확인해 주세요."


# 매개변수의 타입이 BaseModel 이면 FastAPI 는 요청 본문(JSON)을 그 양식으로 검사해 객체로 넘긴다
@app.post("/v1/ask-simple")
def ask_simple(payload: AskRequest):
    """질문을 받아 안내문을 돌려줍니다(응답 양식은 정하지 않았습니다)."""
    # 검사를 통과한 값은 payload.question 처럼 점으로 꺼낸다
    answer = find_answer(payload.question)
    return {
        "answer": answer[: payload.max_length],
        "member_level": payload.member_level,
        "internal_note": "이 값은 밖에 나가면 안 되는 내부 메모입니다",
    }
```

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 로컬 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">내 PC의 실습자료/.venv 커널에서 실행합니다</span></div>

In [ ]:
# question만 담은 본문을 보냅니다(member_level, max_length는 기본값이 채워집니다)
# json= 딕셔너리는 JSON 글자로 바뀌어 요청 본문에 담깁니다(2-4의 params= 는 주소 뒤에 붙습니다)
simple_response = httpx.post(f"{SERVING_URL}/v1/ask-simple", json={"question": "대출은 몇 권까지 되나요?"})
simple_reply = status_and_body(simple_response)
display(simple_reply)

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">member_level</code>이 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">basic</code>:</strong> 보내지 않은 항목에 양식의 기본값이 채워졌습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">internal_note</code>:</strong> 반환 딕셔너리에 담은 내부 메모가 그대로 밖으로 나갔습니다. 이 문제는 3-5에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">response_model</code>로 응답 양식을 정해 막습니다.</li>
</ul>
</div>

### 3-4. 양식이 틀린 요청과 422

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">요청 본문이 양식에 맞지 않으면 FastAPI는 함수를 부르지 않고 422를 돌려줍니다. 쿼리 매개변수의 422와 같은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">detail</code> 모양이며, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">loc</code> 첫 칸이 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">body</code>입니다.</p>
</div>

In [ ]:
# 양식에 맞지 않는 본문 네 가지를 보냅니다
bad_bodies = {
    "question 없음": httpx.post(f"{SERVING_URL}/v1/ask-simple", json={"member_level": "premium"}),
    "max_length=5(10 이상이어야 함)": httpx.post(f"{SERVING_URL}/v1/ask-simple", json={"question": "대출 안내", "max_length": 5}),
    "max_length에 글자": httpx.post(f"{SERVING_URL}/v1/ask-simple", json={"question": "대출 안내", "max_length": "길게"}),
    "질문을 본문이 아니라 쿼리로 보냄": httpx.post(f"{SERVING_URL}/v1/ask-simple", params={"question": "대출 안내"}),
}

# 422 본문의 detail 목록에서 첫 항목의 위치(loc)와 이유(msg)를 꺼내 한 테이블로 봅니다
body_error_rows = []
for label, response in bad_bodies.items():
    first_error = response.json()["detail"][0]
    body_error_rows.append({"보낸 것": label, "상태 코드": response.status_code, "loc": first_error["loc"], "msg": first_error["msg"]})
display(pd.DataFrame(body_error_rows))

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">loc</code>이 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">['body', '항목 이름']</code>:</strong> 본문의 어느 항목이 틀렸는지 알려 줍니다. 마지막 행은 본문 자체가 없어 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">['body']</code>입니다. 같은 이름(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">question</code>)이라도 쿼리로 보내면 본문 양식은 채워지지 않습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">msg</code>:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Field required</code>(필수 항목 없음), <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">greater than or equal to 10</code>(제약 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ge=10</code> 위반), <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">valid integer</code>(정수가 아님)처럼 이유가 나옵니다. 사용자 화면에 이 내용을 보여 주면 무엇을 고칠지 알 수 있습니다.</li>
</ul>
</div>

### 3-5. response_model로 응답 양식 정하기

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1/ask-simple</code>은 반환 딕셔너리에 담은 것이 전부 밖으로 나가므로, 코드를 고치다 내부 값을 하나 더 담으면 그대로 공개됩니다. 데코레이터에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">response_model=응답 양식</code>을 주면 FastAPI가 반환값을 그 양식으로 바꿔 보냅니다. 양식에 없는 항목은 빠지고, 양식에 있는데 빠진 항목은 서버 쪽 오류(500)로 드러나며, 응답 양식이 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/docs</code>에도 표시됩니다.</p>
</div>

```python
# step2_schema.py
class AskResponse(BaseModel):
    """도서관 봇의 답변 한 건."""

    answer: str = Field(description="안내 문장")
    member_level: str = Field(description="요청에 담겨 온 회원 등급")
    length: int = Field(description="답변 글자 수")


# response_model 에 없는 항목(internal_note)은 응답에서 빠진다
@app.post("/v1/ask", response_model=AskResponse)
def ask(payload: AskRequest):
    """질문을 받아 AskResponse 양식의 항목만 돌려줍니다."""
    answer = find_answer(payload.question)[: payload.max_length]
    return {
        "answer": answer,
        "member_level": payload.member_level,
        "length": len(answer),
        "internal_note": "이 값은 응답 양식에 없으므로 밖으로 나가지 않습니다",
    }
```

In [ ]:
# 같은 질문을 응답 양식이 없는 주소와 있는 주소에 보내 응답 항목을 비교합니다
question_body = {"question": "연장 되나요?"}
loose_reply = httpx.post(f"{SERVING_URL}/v1/ask-simple", json=question_body).json()
fixed_reply = httpx.post(f"{SERVING_URL}/v1/ask", json=question_body).json()

# 응답 딕셔너리의 키(항목 이름) 목록을 나란히 봅니다
reply_keys = {"/v1/ask-simple(양식 없음)": list(loose_reply), "/v1/ask(AskResponse)": list(fixed_reply)}
display(reply_keys)

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">/v1/ask</code>의 항목은 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">answer</code>, <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">member_level</code>, <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">length</code> 셋:</strong> 함수는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">internal_note</code>까지 돌려줬지만 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">AskResponse</code>에 없어서 빠졌습니다. 요청 양식은 들어오는 값을, 응답 양식은 나가는 값을 정합니다.</li>
</ul>
<p style="margin:6px 0;">응답에 새 항목을 내보내려면 반환 딕셔너리에 담는 것만으로는 부족합니다. 응답 양식 클래스(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">AskResponse</code>)에도 항목을 더하고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/docs</code>의 응답 예시에 그 항목이 보이는지 확인합니다.</p>
</div>

### 3-6. 🖐️ 함께 따라하기: 짧게 답하기(quiet)

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">요청 양식에 참/거짓 항목 하나를 더하고, 그 값에 따라 답변 길이를 줄이는 주소를 완성합니다. 응답 양식에서 뺀 항목이 응답에서도 빠지는지 함께 확인합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">작성 순서</strong></p>
<ol style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">QuietAskRequest</code>가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">BaseModel</code>을 상속하도록 고치고, 아래 네 필드를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Field</code>로 선언합니다.</li>
</ol>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">필드</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">타입</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">기본값과 제약</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">question</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">str</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">필수, 최소 2자</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">member_level</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">str</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">기본값 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;basic&quot;</code></td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">max_length</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">int</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">기본값 100, 10 이상 500 이하</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">quiet</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">bool</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">기본값 False</td>
</tr>
</tbody>
</table>
<ol start="2">
<li style="margin:3px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">QuietAskResponse</code>도 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">BaseModel</code>을 상속하고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">answer</code>(str), <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">member_level</code>(str), <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">quiet</code>(bool) 세 필드를 선언합니다. 각 필드의 역할을 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Field</code>의 description에 적습니다.</li>
<li style="margin:3px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ask_quiet</code>는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">payload: QuietAskRequest</code>를 받아 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">find_answer</code>로 찾은 답을 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">max_length</code>까지 자릅니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">quiet</code>가 True이면 다시 20자로 줄이고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">answer</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">member_level</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">quiet</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">length</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">internal_note</code>를 담은 딕셔너리를 반환합니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">length</code>는 답변 글자 수, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">internal_note</code>는 내부 메모 문자열입니다.</li>
<li style="margin:3px 0;">함수에 POST <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1/ask-quiet</code>를 등록하고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">response_model</code>을 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">QuietAskResponse</code>로 지정합니다. 반환한 다섯 항목 중 응답에는 양식의 세 항목만 남아야 합니다.</li>
</ol>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">실습 진행 방법:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">step2_schema.py</code> 맨 아래 클래스·함수 이름만 남긴 골격을 완성하고, 아래 요청 셀의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">...</code>를 채웁니다. 참고할 코드는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">AskRequest</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">AskResponse</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ask</code>입니다. 완성본은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">정답/step2_schema.py</code>와 정답 노트북의 같은 절에 있습니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">설정</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">로컬 터미널의 step2 서버는 그대로 둡니다. 파일을 저장하면 서버가 다시 시작합니다.</li>
</ul>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">확인 순서:</strong> 파일을 저장하고 재시작 로그를 확인한 뒤 아래 요청 셀을 실행합니다. 404이면 데코레이터의 메서드·주소를 먼저 확인하고, 422이면 요청 필드·타입·기본값을 확인합니다. 함수 내부 오류는 터미널의 Traceback을 확인합니다.</p>
</div>

```python
# step2_schema.py 맨 아래
# ---------------------------------------------------------------------------
# 🖐️ 함께 따라하기
# ---------------------------------------------------------------------------

# (1) BaseModel을 상속하고 요청 필드 네 개를 작성하세요.
class QuietAskRequest:
    pass


# (2) BaseModel을 상속하고 응답 필드 세 개를 작성하세요.
class QuietAskResponse:
    pass


# (3) POST /v1/ask-quiet의 인자, 데코레이터, 본문을 작성하세요.
def ask_quiet():
    pass
```

<details style="margin:8px 0;"><summary style="font-size:17px;color:#1f3a5f;font-weight:600;cursor:pointer;margin:6px 0;">힌트</summary>

```text
접근방법: 3-2의 AskRequest와 3-5의 AskResponse, ask를 참고합니다.
요청 양식, 응답 양식, 처리 함수 순서로 작성합니다. quiet를 생략할 수 있게 기본값을 정하고, 응답에 내보낼 항목만 응답 양식에 둡니다.
```

</details>

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">확인 기준:</strong> 파일을 저장하면 로컬 터미널에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Application startup complete</code>가 다시 나오고, 오류 줄(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Traceback</code>)이 없습니다.</p>
</div>

In [ ]:
# 🖐️ 함께 따라하기: 두 번째 본문에 첫 본문과 같은 질문과 quiet True를 담으세요
quiet_bodies = {
    "quiet 없음": {"question": "열람실 운영 시간 알려 주세요"},
    "quiet=True": ...,
}

# 본문마다 상태 코드, 답변, 답변 글자 수, 응답 항목을 한 테이블로 봅니다
quiet_rows = []
for label, body in quiet_bodies.items():
    response = httpx.post(f"{SERVING_URL}/v1/ask-quiet", json=body)
    # 200이 아니면 여기서 오류로 멈춥니다
    response.raise_for_status()
    reply = response.json()
    quiet_rows.append({"본문": label, "상태 코드": response.status_code, "답변": reply["answer"], "답변 글자 수": len(reply["answer"]), "응답 항목": list(reply)})
display(pd.DataFrame(quiet_rows))

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">확인 기준:</strong> 두 행 모두 상태 코드 200이고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">quiet=True</code> 행의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">답변 글자 수</code>가 20입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">응답 항목</code>은 두 행 모두 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">answer</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">member_level</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">quiet</code> 셋이며 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">length</code>와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">internal_note</code>가 없습니다.</p>
</div>

### 3-7. 자동 문서(/docs)

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">FastAPI는 앱의 주소, 매개변수, 요청 양식, 응답 양식을 모아 <strong style="font-size:1.13em;color:#1f3a5f;">OpenAPI</strong> 형식(웹 API 명세의 표준 형식)의 API 문서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/openapi.json</code>을 만들고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/docs</code>에서 그 문서를 화면으로 보여 줍니다. 로컬 터미널의 step2 서버를 그대로 두고 문서 화면을 봅니다.</p>
<ol style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">내 PC 브라우저에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">http://127.0.0.1:8080/docs</code>를 엽니다.</li>
<li style="margin:3px 0;">문서 제목이 <strong style="font-size:1.13em;color:#1f3a5f;">도서관 안내 봇 API</strong>인지 확인합니다. 노트북과 브라우저는 같은 PC의 8080 포트에 접속합니다.</li>
<li style="margin:3px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">POST /v1/ask</code>를 펼쳐 <strong style="font-size:1.13em;color:#1f3a5f;">Try it out</strong>을 누르고, 요청 본문의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">question</code>에 질문을 넣어 <strong style="font-size:1.13em;color:#1f3a5f;">Execute</strong>를 누릅니다. 아래 <strong style="font-size:1.13em;color:#1f3a5f;">Response body</strong>에 답이 나옵니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">max_length</code>를 5로 바꿔 다시 보내면 422와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">detail</code>이 나옵니다. <strong style="font-size:1.13em;color:#1f3a5f;">Try it out</strong>을 누르면 그 버튼이 <strong style="font-size:1.13em;color:#1f3a5f;">Cancel</strong>로 바뀌고 요청 본문을 고칠 수 있게 됩니다.</li>
</ol>
<img style="display:block;max-width:100%;height:auto;margin:0;" src="../images/12_docs_ask_try_it_out.png" width="1050" alt="step2 서버의 /docs 화면에서 POST /v1/ask를 펼쳐 Try it out을 누른 뒤 max_length를 5로 보낸 모습입니다. Try it out을 누르면 Cancel로 바뀌는 버튼 자리, Execute 버튼, Response body(422와 detail)에 빨간 사각형을 그렸습니다.">

<ol start="4">
<li style="margin:3px 0;">화면 맨 아래 <strong style="font-size:1.13em;color:#1f3a5f;">Schemas</strong>에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">AskRequest</code>를 펼치면 항목마다 타입, 기본값, 제약, 설명이 보입니다.</li>
</ol>
<p style="margin:6px 0;">이 화면은 코드에서 바로 만들어집니다. 함수 이름은 엔드포인트 제목, docstring은 엔드포인트 설명, 타입 힌트와 기본값은 매개변수와 양식, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Field</code>의 제약과 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">description</code>은 항목 설명이 됩니다. 문서를 따로 쓰지 않아도 코드의 타입 힌트와 설명이 곧 문서가 됩니다.</p>
<p style="margin:6px 0;">노트북에서도 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/docs</code> 화면이 읽는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/openapi.json</code>에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">AskRequest</code> 부분을 꺼내 봅니다.</p>
</div>

In [ ]:
# /docs 화면이 읽는 API 문서(openapi.json)를 받습니다
openapi = httpx.get(f"{SERVING_URL}/openapi.json").json()

# 요청 양식 AskRequest의 항목과 POST /v1/ask의 설명(docstring)을 꺼냅니다
ask_doc = {
    "POST /v1/ask 설명": openapi["paths"]["/v1/ask"]["post"]["description"],
    "AskRequest 필수 항목": openapi["components"]["schemas"]["AskRequest"]["required"],
    "AskRequest 항목": openapi["components"]["schemas"]["AskRequest"]["properties"],
}
display(ask_doc)

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">POST /v1/ask 설명</code>:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ask</code> 함수의 docstring이 그대로 들어갔습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">AskRequest 항목</code>:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">max_length</code>의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">minimum</code> 10.0, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">maximum</code> 500.0(문서에는 소수로 표시됩니다), <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">default</code> 100은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Field(default=100, ge=10, le=500)</code>에서, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">description</code>은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Field</code>의 설명에서 왔습니다. 필수 항목은 기본값이 없는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">question</code> 하나입니다.</li>
</ul>
</div>

## 4. def와 async def

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">비동기 처리(asynchronous processing)</strong>는 기다리는 작업이 끝나기 전에 다른 작업을 처리하는 방식입니다. 서빙 API가 요청 하나에 쓰는 시간은 대부분 계산이 아니라 모델 서버의 답을 기다리는 시간입니다. 기다리는 동안 다른 요청을 처리하면 여러 요청을 함께 끝낼 수 있습니다.</p>
<p style="margin:6px 0;">예를 들어 답을 0.5초씩 기다리는 요청 4건이 들어온다고 해 봅니다. 하나씩 처리하면 0.5초 × 4건 = 2초가 걸립니다. 첫 요청이 기다리는 사이에 둘째, 셋째, 넷째 요청을 시작하면 네 건의 대기 시간이 겹쳐 약 0.5초에 모두 끝납니다. 4-3에서 이 두 경우를 실제로 비교합니다.</p>
</div>

### 4-1. step3_async.py 서버 실행

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · 로컬 터미널</strong></div>

```text
# 앞 단계 서버를 Ctrl+C로 멈춘 뒤, 같은 로컬 터미널에서 실행합니다
uv run uvicorn step3_async:app --host 0.0.0.0 --port 8080 --reload
```

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">로컬 터미널에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Application startup complete</code>가 다시 나오면 새 서버가 요청을 받을 수 있습니다. 로컬 터미널을 새로 열었다면 탐색기에서 교안 폴더의 터미널을 다시 엽니다. 정답 서버를 실행할 때는 이어서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">cd 정답</code>으로 이동합니다.</p>
</div>

### 4-2. 엔드포인트가 실행되는 곳: 이벤트 루프와 스레드 풀

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">코루틴 함수(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">async def</code>로 만든 함수), <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">await</code>, 코루틴들을 번갈아 실행하는 asyncio의 이벤트 루프(event loop)는 에이전트 데이터 분석 자동화 단원의 부록(동기·비동기 기초)과 RAGAS LLM 평가 단원에서 다뤘습니다. FastAPI에서 새로 알아 둘 것은 엔드포인트가 어디에서 실행되는지입니다.</p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">이벤트 루프:</strong> uvicorn은 이벤트 루프 하나에서 모든 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">async def</code> 엔드포인트를 실행합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">스레드 풀(thread pool):</strong> 미리 띄워 둔 작업 스레드(한 프로그램 안에서 따로 실행되는 작업 흐름)의 묶음입니다. FastAPI는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">def</code> 엔드포인트를 이벤트 루프가 아니라 스레드 풀의 스레드(기본 40개)에서 실행합니다.</li>
</ul>
<p style="margin:6px 0;">기다리는 작업은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">asyncio.sleep(초)</code>와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">time.sleep(초)</code>로 만듭니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">await asyncio.sleep</code>은 기다리는 동안 이벤트 루프에 실행을 넘기고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">time.sleep</code>은 부른 스레드를 그 시간 동안 멈춥니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">await run_in_threadpool(함수, 인자)</code>는 동기 함수를 스레드 풀의 스레드에서 실행하고, 이벤트 루프는 그동안 다른 요청을 처리합니다.</p>
</div>

```python
# step3_async.py
# 모델 서버의 답을 기다리는 시간 대신 쓰는 대기 시간(초)
BACKEND_DELAY = 0.5


# def 엔드포인트: FastAPI 가 스레드 풀(미리 띄워 둔 작업 스레드 묶음)의 스레드 하나에서 실행한다
@app.get("/wait/def")
def wait_def():
    """일반 함수 엔드포인트. time.sleep 으로 BACKEND_DELAY 초 동안 이 스레드만 멈춥니다."""
    time.sleep(BACKEND_DELAY)
    return {"endpoint": "def + time.sleep"}


# async def 엔드포인트: 이벤트 루프에서 실행하고, await 에서 기다리는 동안 다른 요청을 처리한다
@app.get("/wait/async")
async def wait_async():
    """코루틴 엔드포인트. await asyncio.sleep 으로 기다리는 동안 이벤트 루프를 비워 둡니다."""
    await asyncio.sleep(BACKEND_DELAY)
    return {"endpoint": "async def + await asyncio.sleep"}


# 잘못 쓴 예: async def 안의 time.sleep 은 이벤트 루프 자체를 멈춰, 그동안 다른 요청을 하나도 처리하지 못한다
@app.get("/wait/blocking")
async def wait_blocking():
    """코루틴 엔드포인트 안에서 time.sleep 을 부른 예. 이벤트 루프가 BACKEND_DELAY 초 동안 멈춥니다."""
    time.sleep(BACKEND_DELAY)
    return {"endpoint": "async def + time.sleep"}


# 고친 예: async def 안의 동기 함수는 run_in_threadpool 로 스레드 풀에 넘긴다
@app.get("/wait/thread")
async def wait_thread():
    """time.sleep 을 스레드에서 실행하고, 이벤트 루프는 기다리는 동안 다른 요청을 처리합니다."""
    await run_in_threadpool(time.sleep, BACKEND_DELAY)  # 함수와 인자를 따로 넘긴다
    return {"endpoint": "async def + run_in_threadpool(time.sleep)"}
```

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">주소</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">실행하는 곳</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">기다리는 동안 다른 요청</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">GET /wait/def</code>(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">def</code> + <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">time.sleep</code>)</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">스레드 풀의 스레드</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">처리합니다(멈춘 것은 그 스레드 하나입니다).</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">GET /wait/async</code>(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">async def</code> + <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">await asyncio.sleep</code>)</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">이벤트 루프</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">처리합니다(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">await</code>에서 이벤트 루프가 다른 요청을 실행합니다).</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">GET /wait/blocking</code>(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">async def</code> + <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">time.sleep</code>)</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">이벤트 루프</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">처리하지 못합니다(이벤트 루프가 멈춥니다).</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">GET /wait/thread</code>(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">async def</code> + <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">await run_in_threadpool(time.sleep, ...)</code>)</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">대기는 스레드 풀의 스레드, 엔드포인트는 이벤트 루프</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">처리합니다(멈춘 것은 그 스레드 하나입니다).</td>
</tr>
</tbody>
</table>
<img style="display:block;max-width:100%;height:auto;margin:0;" src="../images/09_def_async_timeline.png" width="1050" alt="요청 4건이 기다리는 방식을 네 줄의 시간축으로 비교한 그림입니다. def와 time.sleep은 스레드 4개가 대기를 나란히 겹치고, async def와 await asyncio.sleep은 이벤트 루프 하나 위에서 대기 4개가 겹쳐 함께 끝나며, async def 안의 time.sleep은 이벤트 루프를 멈춰 대기 4개가 차례로 이어집니다. async def 안에서 run_in_threadpool로 넘기면 대기 4개가 스레드 4개에서 다시 나란히 겹칩니다.">
</div>

### 4-3. 요청 4건을 동시에 보내 걸린 시간 비교

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">네 주소에 요청을 4건씩 동시에 보내, 4건이 모두 끝날 때까지 걸린 시간을 비교합니다. 노트북이 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">httpx.get</code>으로 하나씩 보내면 앞 요청의 답이 와야 다음 요청이 출발하므로, 서버를 어떻게 만들었든 네 주소 모두 약 2초가 걸립니다. 서버가 기다리는 동안 다른 요청을 처리하는지 보려면 요청 4건이 한꺼번에 서버에 닿아야 합니다. 그래서 노트북도 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">httpx.AsyncClient</code></strong>(httpx의 비동기 클라이언트)로 요청을 동시에 보냅니다. 주피터 노트북에서는 셀 맨 바깥에서 바로 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">await</code>를 쓸 수 있습니다. RAGAS LLM 평가 단원에서처럼 세마포어와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">run_limited</code>로 동시 실행 수를 묶고, gather로 함께 기다립니다.</p>
</div>

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 로컬 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">내 PC의 실습자료/.venv 커널에서 실행합니다</span></div>

In [ ]:
import asyncio
import time

# 동시에 보낼 요청 수. 요청 4건을 한꺼번에 보내 네 주소의 걸린 시간을 비교하려고 4로 둡니다
MAX_CONCURRENCY = 4
semaphore = asyncio.Semaphore(MAX_CONCURRENCY)  # 동시에 실행하는 작업 수를 MAX_CONCURRENCY 개까지로 제한합니다


async def run_limited(task):
    """세마포어 자리가 날 때까지 기다렸다가 작업 하나를 실행합니다."""
    async with semaphore:
        return await task


async def send_together(path, count):
    """같은 주소에 GET 요청 count 건을 동시에 보내고, 모두 끝날 때까지 걸린 시간(초)을 돌려줍니다."""
    async with httpx.AsyncClient(base_url=SERVING_URL, timeout=30) as client:
        started = time.perf_counter()
        tasks = [run_limited(client.get(path)) for _ in range(count)]
        responses = await asyncio.gather(*tasks)  # 함께 시작해 모두 끝날 때까지 기다리고, 넘긴 순서대로 결과를 돌려줍니다
        elapsed = time.perf_counter() - started

    # 200이 아닌 응답이 있으면 여기서 오류로 멈춥니다
    for response in responses:
        response.raise_for_status()
    return round(elapsed, 2)

In [ ]:
# 네 주소에 요청 4건씩을 동시에 보내 걸린 시간을 한 테이블로 모읍니다
wait_paths = {
    "/wait/def": "def + time.sleep",
    "/wait/async": "async def + await asyncio.sleep",
    "/wait/blocking": "async def + time.sleep",
    "/wait/thread": "async def + run_in_threadpool(time.sleep)",
}
wait_rows = []
for path, kind in wait_paths.items():
    seconds = await send_together(path, MAX_CONCURRENCY)
    wait_rows.append({"주소": path, "엔드포인트": kind, "동시 요청 수": MAX_CONCURRENCY, "4건이 모두 끝난 시간(초)": seconds})
display(pd.DataFrame(wait_rows))

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">/wait/def</code>와 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">/wait/async</code>는 약 0.5~0.8초:</strong> 한 건의 대기 시간(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">BACKEND_DELAY</code> 0.5초)이 4건에서 겹쳤습니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">def</code> 엔드포인트는 요청마다 스레드 풀의 다른 스레드에서 실행되므로, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">time.sleep</code>이 멈추는 것은 그 스레드 하나뿐입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">/wait/blocking</code>은 약 2초(0.5초 × 4건) 이상:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">async def</code> 안의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">time.sleep</code>이 이벤트 루프를 멈춰, 4건을 하나씩 차례로 처리했습니다. 소수 둘째 자리는 실행마다 조금씩 다릅니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">/wait/thread</code>는 다시 약 0.5~0.8초:</strong> 같은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">time.sleep</code>을 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">run_in_threadpool</code>로 스레드 풀에 넘기자, 멈추는 것은 스레드 하나뿐이고 이벤트 루프는 4건을 함께 처리했습니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">def</code> 엔드포인트에서 FastAPI가 자동으로 해 주는 일을 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">async def</code> 안에서 직접 한 것입니다.</li>
</ul>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">/wait/blocking</code>이 흔한 실수입니다.</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">async def</code> 엔드포인트 안에서 동기 함수(끝날 때까지 실행을 넘기지 않는 함수. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">time.sleep</code>, 동기 HTTP 클라이언트, 동기 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">OpenAI</code> 클라이언트)를 부르면 기다리는 동안 이벤트 루프가 멈추고, 그 요청이 끝날 때까지 서버 전체가 다른 요청을 받지 못합니다. 그래서 교안 01의 서빙 API는 vLLM에 요청할 때 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">await</code>로 기다리는 비동기 클라이언트 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">AsyncOpenAI</code>를 씁니다.</p>
<p style="margin:6px 0;">엔드포인트는 이렇게 고릅니다. 기다리는 작업을 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">await</code>로 부를 수 있으면(비동기 라이브러리) <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">async def</code>로, 동기 라이브러리만 있으면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">def</code>로 둡니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">async def</code> 안에서 동기 함수를 꼭 불러야 하면 그 호출만 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">await run_in_threadpool(...)</code>로 스레드에 넘깁니다. 다만 스레드 풀은 기본 40개라 그보다 많이 동시에 기다리면 줄을 서고, 비동기 판이 있는 라이브러리(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">httpx.AsyncClient</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">AsyncOpenAI</code> 등)는 스레드 없이 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">await</code>로 바로 쓰는 쪽이 가볍습니다.</p>
<p style="margin:6px 0;">동시 처리로 시간이 줄어드는 것은 기다리는 작업(모델 서버 호출, 다른 API 호출, 데이터베이스 조회)이 겹칠 때뿐입니다. 큰 배열 계산처럼 기다림 없이 계산만 하는 작업은 겹칠 틈이 없어서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">async def</code>로 바꿔도 빨라지지 않습니다.</p>
</div>

## 5. 스트리밍 응답

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">스트리밍(streaming)</strong>은 답 전체가 끝나기 전에 만들어진 조각부터 보내는 응답 방식입니다. 언어 모델은 답을 앞에서부터 조금씩 만들므로, 조각으로 보내면 사용자는 답 전체를 기다리지 않고 첫 조각부터 읽습니다. 답 전체가 끝나는 시각은 한 번에 보낼 때와 거의 같고, 앞당겨지는 것은 첫 조각이 보이는 시각입니다.</p>
</div>

### 5-1. step4_streaming.py 서버 실행

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · 로컬 터미널</strong></div>

```text
# 앞 단계 서버를 Ctrl+C로 멈춘 뒤, 같은 로컬 터미널에서 실행합니다
uv run uvicorn step4_streaming:app --host 0.0.0.0 --port 8080 --reload
```

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">로컬 터미널에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Application startup complete</code>가 다시 나오면 새 서버가 요청을 받을 수 있습니다. 로컬 터미널을 새로 열었다면 탐색기에서 교안 폴더의 터미널을 다시 엽니다. 정답 서버를 실행할 때는 이어서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">cd 정답</code>으로 이동합니다.</p>
</div>

### 5-2. 비동기 제너레이터와 StreamingResponse

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">제너레이터(generator):</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">return</code> 대신 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">yield</code>로 값을 하나씩 내놓는 함수입니다. 값을 하나 내놓으면 그 자리에서 멈춰 있다가, 다음 값을 요청받으면 이어서 실행합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">비동기 제너레이터(async generator):</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">async def</code> 안에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">yield</code>를 쓰는 함수입니다. 조각과 조각 사이에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">await</code>로 기다릴 수 있고, 기다리는 동안 이벤트 루프는 다른 요청을 처리합니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">async for</code>로 값을 하나씩 꺼냅니다.</li>
</ul>
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">find_answer</code>는 안내문 딕셔너리에서 답을 고릅니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">answer_pieces</code>는 안내문을 낱말 하나씩 조각으로 내놓고, 비교용 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1/ask-once</code>는 같은 조각을 끝까지 모아 JSON 하나로 돌려줍니다.</p>
</div>

```python
# step4_streaming.py
# 모델이 다음 조각을 만드는 데 걸리는 시간 대신 쓰는 대기 시간(초)
PIECE_DELAY = 0.3


class StreamRequest(BaseModel):
    """스트리밍으로 답변을 받을 질문 한 건."""

    question: str = Field(min_length=2, description="이용자의 질문 문장")


async def answer_pieces(question: str):
    """안내문을 낱말 하나씩 조각으로 내놓는 비동기 제너레이터입니다."""
    text = find_answer(question)
    for word in text.split(" "):
        # 다음 조각이 만들어지기를 기다린다. 기다리는 동안 이벤트 루프는 다른 요청을 처리한다
        await asyncio.sleep(PIECE_DELAY)
        # 낱말 뒤에 띄어쓰기를 붙여 내놓는다. 받는 쪽은 조각을 이어 붙이면 원래 문장이 된다
        yield f"{word} "


# StreamingResponse 는 제너레이터가 조각을 내놓을 때마다 그 조각을 바로 보낸다
# 응답이 딕셔너리 하나가 아니라 이어 보내는 글자 조각이라 response_model 은 쓰지 않는다(요청 양식 검사는 그대로 한다)
@app.post("/v1/ask-stream")
async def ask_stream(payload: StreamRequest):
    """답변을 조각으로 이어 보냅니다(text/plain)."""
    # media_type="text/plain" 은 일반 글자를 보낸다는 표시다
    return StreamingResponse(answer_pieces(payload.question), media_type="text/plain")


# 비교용: 같은 조각을 끝까지 모은 뒤 한 번에 돌려준다
@app.post("/v1/ask-once")
async def ask_once(payload: StreamRequest):
    """답변을 끝까지 만든 뒤 JSON 하나로 돌려줍니다."""
    text = ""
    async for piece in answer_pieces(payload.question):
        text += piece
    # 마지막 조각 뒤의 띄어쓰기는 뺀다
    return {"answer": text.strip()}
```

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">상태 코드는 첫 조각 전에 정해집니다:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">StreamingResponse</code>는 상태 코드 200과 헤더(header)를 먼저 보내고 조각을 이어 보냅니다. 그 뒤에는 상태 코드를 바꿀 수 없으므로, 요청 검사(422)와 실패 판단은 첫 조각을 보내기 전에 끝냅니다.</li>
</ul>
</div>

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">스트리밍 요청과 조각 도착 시각</strong></p>
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">read_pieces</code>는 함께 따라하기에서도 쓰므로 함수로 둡니다.</p>
</div>

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 로컬 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">내 PC의 실습자료/.venv 커널에서 실행합니다</span></div>

In [ ]:
def read_pieces(path, question):
    """스트리밍 주소에 질문을 보내고, 받은 조각마다 순번과 도착 시각(초)을 붙여 리스트로 돌려줍니다."""
    rows = []
    sent_at = time.perf_counter()
    # httpx.stream 은 응답 본문을 한꺼번에 받지 않고 도착하는 대로 읽게 합니다
    with httpx.stream("POST", f"{SERVING_URL}{path}", json={"question": question}) as response:
        # 200이 아니면 조각을 읽기 전에 오류로 멈춥니다
        response.raise_for_status()
        # iter_text()는 도착한 본문을 글자로 바꿔 받은 만큼씩 내놓습니다. 도착 시각은 요청을 보낸 뒤 걸린 초입니다
        for piece in response.iter_text():
            rows.append({"순번": len(rows) + 1, "도착 시각(초)": round(time.perf_counter() - sent_at, 2), "조각": piece})
    return rows

In [ ]:
# 대출 질문을 스트리밍 주소로 보내 조각이 도착하는 시각을 봅니다
loan_question = "대출은 몇 권까지 되나요?"
stream_rows = read_pieces("/v1/ask-stream", loan_question)
display(pd.DataFrame(stream_rows))

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">순번</code>과 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">조각</code>:</strong> 안내문의 낱말 7개가 한 행씩, 보낸 순서대로 도착했습니다. 조각을 이어 붙이면 원래 안내문입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">도착 시각(초)</code>:</strong> 약 0.3초(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">PIECE_DELAY</code>) 간격으로 늘어납니다. 서버가 조각을 하나 만들 때마다 바로 보냈다는 뜻입니다.</li>
</ul>
</div>

In [ ]:
# 같은 질문을 한 번에 받는 주소로 보냅니다
sent_at = time.perf_counter()
once_response = httpx.post(f"{SERVING_URL}/v1/ask-once", json={"question": loan_question})
once_seconds = round(time.perf_counter() - sent_at, 2)

# 전체 답이 도착한 시각을 스트리밍의 첫 조각, 마지막 조각 도착 시각과 비교합니다
arrival_compare = {
    "한 번에 받기: 전체 답 도착(초)": once_seconds,
    "스트리밍: 첫 조각 도착(초)": stream_rows[0]["도착 시각(초)"],
    "스트리밍: 마지막 조각 도착(초)": stream_rows[-1]["도착 시각(초)"],
    "한 번에 받은 답": once_response.json()["answer"],
}
display(arrival_compare)

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">전체 답 도착과 마지막 조각 도착이 약 2.1초로 비슷합니다:</strong> 스트리밍이 답 전체를 빨리 끝내지는 않습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">첫 조각은 약 0.3~0.4초:</strong> 한 번에 받는 방식은 2초 넘게 아무것도 받지 못하고, 스트리밍은 첫 조각이 오자마자 화면에 글자를 띄울 수 있습니다. 그 뒤 조각은 약 0.3초 간격으로 옵니다.</li>
</ul>
<p style="margin:6px 0;">스트리밍은 챗봇 답변이나 요약처럼 사람이 읽는 긴 글에 잘 맞습니다. 분류 라벨이나 점수처럼 다른 프로그램이 다 받은 뒤에 쓰는 짧은 결과는 JSON 하나로 돌려주는 편이 다루기 쉽습니다.</p>
</div>

### 5-3. lifespan: DB 연결 준비와 종료

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">lifespan(앱 수명 주기)</strong>은 서버 프로세스가 시작할 때 공유 자원을 준비하고 종료할 때 정리하는 처리입니다. DB 연결을 요청마다 새로 열지 않고, 서버가 뜰 때 한 번 열어 앱 전체가 함께 쓰다가 서버를 끌 때 닫습니다.</p>
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">step5_lifespan.py</code>는 파이썬에 들어 있는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">sqlite3</code>로 같은 폴더의 SQLite 파일 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">library.db</code>에 연결합니다. SQLite는 DB 전체가 파일 하나라서, 가입하거나 DB 서버를 띄우지 않고 바로 씁니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">library.db</code>의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">books</code> 테이블에는 책 5권이 들어 있고, 12번과 45번은 2-3의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">BOOKS</code>와 같은 책입니다(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">available</code>의 1은 대출 가능, 0은 대출 중).</p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">책 번호(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">book_id</code>)</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">제목(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">title</code>)</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">서가(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shelf</code>)</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">대출 가능(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">available</code>)</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">12</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">파이썬으로 배우는 데이터 분석</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">A-3</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">1</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">23</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">SQL 첫걸음</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">B-4</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">1</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">45</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">도서관 이용 안내서</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">B-1</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">0</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">58</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">웹 API 설계 입문</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">C-1</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">0</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">77</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">FastAPI로 만드는 서빙 API</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">C-2</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">1</td>
</tr>
</tbody>
</table>
</div>

```python
import sqlite3
from contextlib import asynccontextmanager
from pathlib import Path
from fastapi import FastAPI, HTTPException
```

```python
# step5_lifespan.py
# 이 파일과 같은 폴더의 SQLite 파일. books 테이블에 책 번호, 제목, 서가, 대출 가능 여부가 들어 있다.
DB_PATH = Path(__file__).resolve().parent / "library.db"


# @asynccontextmanager 를 붙이면 yield 앞은 시작 때 준비 코드, 뒤(finally)는 종료 때 정리 코드로 쓰인다
@asynccontextmanager
async def lifespan(app: FastAPI):
    # 프로세스 시작 때 DB 파일에 한 번 연결해 앱 전체가 함께 쓴다.
    # def 엔드포인트는 스레드 풀의 다른 스레드에서 실행되므로 check_same_thread=False 로 허용한다(이 예시는 읽기만 한다).
    connection = sqlite3.connect(DB_PATH, check_same_thread=False)
    connection.row_factory = sqlite3.Row  # 조회 결과의 행을 열 이름으로 읽는다(row["title"]).
    app.state.db = connection  # app.state 는 앱 전체가 함께 쓸 값을 두는 공간이다
    print("DB 연결 준비 완료")
    try:
        yield  # 준비가 끝나면 FastAPI가 요청을 처리한다.
    finally:
        connection.close()  # 이 앱의 연결을 닫는다. DB 파일은 그대로 남는다.
        print("DB 연결 종료 완료")


app = FastAPI(title="SQLite 연결 수명 주기", lifespan=lifespan)


# sqlite3 는 동기 라이브러리라 def 로 작성한다(스레드 풀에서 실행된다)
@app.get("/db/books/{book_id}")
def db_book(book_id: int):
    """앱이 시작할 때 연 연결로 books 테이블에서 책 한 권을 찾는다."""
    # ? 자리에 값을 따로 넘겨, 사용자 입력을 SQL 문자열에 직접 붙이지 않는다.
    row = app.state.db.execute(
        "SELECT book_id, title, shelf, available FROM books WHERE book_id = ?", (book_id,)
    ).fetchone()
    if row is None:
        raise HTTPException(status_code=404, detail=f"{book_id}번 책이 없습니다.")
    return {"book_id": row["book_id"], "title": row["title"], "shelf": row["shelf"], "available": bool(row["available"])}
```

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">step4 서버를 Ctrl+C로 멈추고 같은 로컬 터미널에서 실행합니다.</p>
</div>

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · 로컬 터미널</strong></div>

```text
uv run uvicorn step5_lifespan:app --host 0.0.0.0 --port 8080 --reload
```

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">DB 연결 준비 완료</code>와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Application startup complete</code>가 나오면 아래 요청을 보냅니다.</p>
</div>

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 로컬 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">내 PC의 실습자료/.venv 커널에서 실행합니다</span></div>

In [ ]:
# 같은 연결을 쓰는 책 조회 주소로 12번, 45번 책을 차례로 요청합니다
db_rows = []
for book_id in [12, 45]:
    response = httpx.get(f"{SERVING_URL}/db/books/{book_id}", timeout=15)
    response.raise_for_status()
    db_rows.append({"상태 코드": response.status_code, **response.json()})
display(pd.DataFrame(db_rows))

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">확인 기준:</strong> 두 요청 모두 200이고, 12번은 서가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">A-3</code>·대출 가능(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">True</code>), 45번은 서가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">B-1</code>·대출 중(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">False</code>)으로 2-3의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">BOOKS</code>와 같습니다. 요청마다 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">DB 연결 준비 완료</code>가 다시 찍히지 않습니다. 다음 서버로 바꿀 때 Ctrl+C를 누르면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">DB 연결 종료 완료</code>가 찍힙니다.</p>
</div>

### 5-4. LangChain 모델 응답을 API로 중계

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">이번에는 정해 둔 안내문 대신 <strong style="font-size:1.13em;color:#1f3a5f;">LangChain의 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">ChatOpenAI</code>가 모델 API에서 받은 응답을 스트리밍으로 중계</strong>합니다. 웹 화면이나 다른 프로그램은 모델 API를 직접 호출하지 않고 우리 FastAPI 주소를 호출합니다.</p>
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">실습자료/.env</code>에 앞 단원에서 사용한 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">OPENAI_API_KEY</code>가 있어야 합니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">OPENAI_CHAT_MODEL</code>을 지정하지 않으면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">gpt-4.1-mini</code>를 사용합니다. 실제 모델 호출이므로 API 사용량이 발생합니다.</p>
</div>

```python
import httpx
from langchain_openai import ChatOpenAI
```

```python
# step6_langchain_stream.py
@asynccontextmanager
async def lifespan(app: FastAPI):
    # 모든 비동기 모델 요청이 같은 HTTP 연결풀을 재사용한다.
    http_client = httpx.AsyncClient(timeout=60)
    try:
        app.state.model = ChatOpenAI(
            model=os.environ.get("OPENAI_CHAT_MODEL", "gpt-4.1-mini"),
            http_async_client=http_client,
            temperature=0,
            max_tokens=256,
        )
        yield
    finally:
        await http_client.aclose()  # 앱 종료 시 공유 HTTP 연결도 닫는다.


app = FastAPI(title="LangChain 스트리밍 API", lifespan=lifespan)


class ChatRequest(BaseModel):
    question: str = Field(min_length=2, max_length=500, description="모델에게 보낼 질문")


async def chat_pieces(question: str):
    """모델이 보내는 텍스트를 모으지 않고 받는 즉시 내놓는다."""
    messages = [("system", "질문에 한국어로 세 문장 이내로 답해야 한다."), ("human", question)]
    # astream 이 내놓는 AIMessageChunk 마다 텍스트(chunk.text)만 꺼내 받는 즉시 내놓는다
    async for chunk in app.state.model.astream(messages):
        if chunk.text:
            yield chunk.text


@app.post("/v1/chat/stream")
async def chat_stream(payload: ChatRequest):
    return StreamingResponse(chat_pieces(payload.question), media_type="text/plain")
```

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">step5 서버를 Ctrl+C로 멈추고 같은 로컬 터미널에서 실행합니다.</p>
</div>

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · 로컬 터미널</strong></div>

```text
uv run uvicorn step6_langchain_stream:app --host 0.0.0.0 --port 8080 --reload
```

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">모델 응답을 요청하는 코드는 노트북이 아닌 서버 파일에 있습니다. 아래 노트북 함수는 우리 서버가 보내는 텍스트를 도착하는 대로 화면에 표시하고, 도착 시각도 기록합니다. 함께 따라하기에서도 사용합니다.</p>
</div>

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 로컬 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">내 PC의 실습자료/.venv 커널에서 실행합니다</span></div>

In [ ]:
def read_model_stream(path, payload):
    """우리 API에서 받은 텍스트를 즉시 출력하고 조각별 도착 시각을 기록합니다."""
    rows = []
    full_text = ""
    started = time.perf_counter()
    with httpx.stream("POST", f"{SERVING_URL}{path}", json=payload, timeout=60) as response:
        response.raise_for_status()
        for piece in response.iter_text():
            print(piece, end="", flush=True)
            full_text += piece
            rows.append({"도착 시각(초)": round(time.perf_counter() - started, 2), "텍스트": piece})
    print()
    return full_text, rows


# 노트북 -> 우리 FastAPI -> LangChain -> 모델 API 순서로 요청합니다.
chat_text, chat_rows = read_model_stream(
    "/v1/chat/stream", {"question": "FastAPI 스트리밍 응답의 장점을 두 문장으로 설명해 주세요."}
)

In [ ]:
# 모델이 생성한 텍스트가 도착하는 시각을 확인합니다.
display(pd.DataFrame(chat_rows))

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기:</strong> 모델이 생성한 글이 완성되기를 기다리지 않고 앞부분부터 표시됩니다. 조각의 수와 문구는 실행마다 달라지며, 네트워크에서 여러 모델 조각이 한 수신 행으로 합쳐질 수 있습니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">text/plain</code> 응답이므로 클라이언트는 받은 문자열을 이어 붙이면 전체 답이 됩니다.</p>
</div>

### 5-5. 🖐️ 함께 따라하기: LangChain 요약 API 스트리밍

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">글을 보내면 LangChain으로 한 문장 요약을 만들고, 그 텍스트를 우리 API의 스트리밍 응답으로 돌려줍니다. 모델과 HTTP 클라이언트는 본실습의 lifespan에서 만든 것을 그대로 씁니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">작성 순서</strong></p>
<ol style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">SummaryRequest</code>가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">BaseModel</code>을 상속하도록 고치고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">text</code> 필드를 선언합니다. 타입은 str, 필수 항목이며 길이는 10~2000자입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Field</code>에 역할 설명도 적습니다.</li>
<li style="margin:3px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">summary_pieces</code>가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">text: str</code>을 받도록 고칩니다. 모델에 “주어진 글을 한국어 한 문장으로 요약해야 한다”는 system 메시지와 입력 글을 담은 human 메시지를 보냅니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">app.state.model.astream()</code>으로 조각을 받으며, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">chunk.text</code>가 있을 때마다 그대로 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">yield</code>합니다.</li>
<li style="margin:3px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">summarize_stream</code>은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">payload: SummaryRequest</code>를 받습니다. POST <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1/summary/stream</code>을 등록하고, 요약 제너레이터를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">StreamingResponse</code>로 반환합니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">media_type</code>은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;text/plain&quot;</code>입니다.</li>
</ol>
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">step6_langchain_stream.py</code> 맨 아래 골격에서 상속·인자·데코레이터·본문을 작성합니다. 참고할 코드는 바로 위 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ChatRequest</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">chat_pieces</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">chat_stream</code>입니다. 서버는 그대로 두고 저장 후 재시작 로그를 확인합니다. 404이면 주소 등록, 422이면 필드와 제약을 확인합니다.</p>
</div>

```python
# step6_langchain_stream.py 맨 아래
# ---------------------------------------------------------------------------
# 🖐️ 함께 따라하기
# ---------------------------------------------------------------------------

# (1) BaseModel을 상속하고 요약할 text 필드를 작성하세요.
class SummaryRequest:
    pass


# (2) LangChain으로 요약을 요청하고 텍스트 조각을 그대로 내놓으세요.
async def summary_pieces():
    pass


# (3) 요약 요청을 받아 텍스트 스트림으로 반환하는 주소를 등록하세요.
async def summarize_stream():
    pass
```

<details style="margin:8px 0;"><summary style="font-size:17px;color:#1f3a5f;font-weight:600;cursor:pointer;margin:6px 0;">힌트</summary>

```text
접근방법: 5-4의 ChatRequest, chat_pieces, chat_stream을 참고합니다.
클래스, 비동기 제너레이터, 엔드포인트 순서로 작성합니다. 모델 결과 전체를 모은 뒤 반환하지 말고, 텍스트 조각을 받는 즉시 내놓습니다.
```

</details>

In [ ]:
# 직접 만든 요약 API에 글을 보내고, 요약이 생성되는 대로 읽습니다.
summary_source = (
    "도서관은 이번 달부터 평일 운영 시간을 오후 8시에서 오후 10시까지 연장합니다. "
    "주말 운영 시간은 기존과 같으며, 연장 시간에는 대출과 반납 서비스를 이용할 수 있습니다."
)
summary_text, summary_rows = read_model_stream("/v1/summary/stream", {"text": summary_source})

In [ ]:
# 받은 텍스트와 조각별 도착 시각을 확인합니다.
display(pd.DataFrame(summary_rows))

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">확인 기준:</strong> 입력 글의 핵심인 평일 운영 시간 연장이 요약에 포함되고, 생성 중인 텍스트가 앞부분부터 표시됩니다. 짧은 글이므로 한 번에 받은 것처럼 보일 수도 있습니다. 모델 호출을 숨기지 않고 우리 서버가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">text/plain</code> 조각으로 중계하는 코드인지 확인합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">서버 종료:</strong> 실습을 마치면 로컬 터미널에서 Ctrl+C를 눌러 서버를 종료합니다. lifespan의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">finally</code>에서 공유 HTTP 연결도 닫힙니다.</p>
</div>